In [1]:
import sqlite3
import pandas as pd
import os
import numpy as np
import re
from estnltk import Text

In [9]:
MAARUS_STATUS = 'mitte kunagi'
maarus_status2 = MAARUS_STATUS.replace(" ", "_")
DEPREL = 'obl'
pat_table_name = f"patterns_isikud_len1_{maarus_status2}" # lõpptabel, va ühes cellis, kus on vaja käsitsi muuta

In [3]:
# lahti pakitud märgenduste andmed
root = "data_files"
margendused = pd.read_csv(os.path.join(root,"../data_files/every_verb_case_obl.csv"), sep=";", encoding="utf-8")
margendused

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
0,saama - abl (kellelt/millelt),saama,abl (kellelt/millelt),vahel,vahel,mitte kunagi
1,tulema - abl (kellelt/millelt),tulema,abl (kellelt/millelt),vahel,vahel,mitte kunagi
2,küsima - abl (kellelt/millelt),küsima,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
3,nõudma - abl (kellelt/millelt),nõudma,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
4,võtma - abl (kellelt/millelt),võtma,abl (kellelt/millelt),vahel,vahel,mitte kunagi
...,...,...,...,...,...,...
10574,musitseerima - in (kelles/milles),musitseerima,in (kelles/milles),mitte kunagi,alati,muu
10575,kõigutama - in (kelles/milles),kõigutama,in (kelles/milles),mitte kunagi,mitte kunagi,muu
10576,kätlema - in (kelles/milles),kätlema,in (kelles/milles),mitte kunagi,alati,muu
10577,kõmmutama - in (kelles/milles),kõmmutama,in (kelles/milles),mitte kunagi,alati,mitte kunagi


In [4]:
margendused.case.unique()

array(['abl (kellelt/millelt)',
       'adit (kellesse/millesse lühike nt külla/keelde/vette/majja)',
       'ad (kellel/millel)', 'maldama', 'all (kellele/millele)',
       'el (kellest/millest)', 'ill (kellesse/millesse)',
       'in (kelles/milles)'], dtype=object)

In [10]:
def process_verbs(df):
    # verbide tükeldamine (pea)verbiks ja selle muudeks osadeks
    verb_word = []
    compound_prt1 = []
    compound_prt2 = []
    compound_prt3 = []

    for idx, row in df.iterrows():
        verb = ''
        compound = ['', '', ''] # in current data, there are max 3 compound pieces
        pieces = row['word'].strip().split()

        j = len(pieces)-1 # (pea)verbi leidmine. Oletus on, et pikema konstruktsiooni viimane verbist liige on peaverb
        while j >= 0:
            text = Text(pieces[j]).tag_layer('morph_analysis')
            if 'V' in text.morph_analysis.partofspeech[0]:
                verb = pieces[j]
                pieces.pop(j)
                break
            j-=1

        for i in range(len(pieces)): # allesjäänud jupid määratakse konstruktsiooni ülejäänud osadeks
            compound[i] = pieces[i]

        verb_word.append(verb)
        compound_prt1.append(compound[0])
        compound_prt2.append(compound[1])
        compound_prt3.append(compound[2])
        
    return verb_word, compound_prt1, compound_prt2, compound_prt3

In [13]:
def process_isikumaarused(maarus_status, deprel):
    
    # salvesta isikumäärused
    isikumaarused = margendused[margendused["isikumäärus"]==maarus_status]
    isikumaarused = isikumaarused.reset_index().drop("index", axis=1)
    #isikumaarused.to_csv(f"isikumaarused_{maarus_status}.csv", sep=",", encoding="utf-8")
    
    # lüüa lahku juhud kus verbile on antud komaga eraldatuna kaks võimalikku kaassõna 
    # nt olema kokku, võlgu -> olema kokku ja olema võlgu

    uus1 = []
    for i in range(len(isikumaarused)):
        verb = isikumaarused.iloc[i]["verb"].strip()
        case_osad =  isikumaarused.iloc[i]["case"].strip().split(" ")
        if len(case_osad)>=2:
            case = case_osad[0].strip()
            gov = case_osad[1].replace("(", "").replace(")", "").strip()
            if "," in verb:
                osad = verb.split(" ")
                v = osad[0].strip()
                v1 = osad[1].replace(",", "").strip()
                v2 = osad[2].replace(",", "").strip()
                if len(osad)>3:
                    print("rohkem osasid!")

                uus1.append((isikumaarused.iloc[i]["verbobl"], v+" "+v1, case, gov))
                uus1.append((isikumaarused.iloc[i]["verbobl"], v+" "+v2, case, gov))
            else:
                uus1.append((isikumaarused.iloc[i]["verbobl"], isikumaarused.iloc[i]["verb"], case, gov))

    df1 = pd.DataFrame(uus1, columns=["verbobl", "word", "case", "gov"])
    #df1.to_csv(f"isikumaarused_{maarus_status}.csv", sep=",", encoding="utf-8")
    
    verb_word, comp1, comp2, comp3 = process_verbs(df1)
    df1['verb_word'] = verb_word
    df1['compound_prt1'] = comp1
    df1['compound_prt2'] = comp2
    df1['compound_prt3'] = comp3
    
    df1.insert(0, 'pat_id', range(0, 0 + len(df1)))
    df1.insert(9, 'adp', '')
    df1.insert(10, 'other', '')
    df1.insert(11, 'verb', '')
    df1.insert(12, 'deprel', deprel)
    maarus_status2 = maarus_status.replace(" ", "_")
    df1.to_csv(f"isikumaarused_{maarus_status2}_verb_patterns.csv", sep=",", encoding="utf-8", index = False)
    
    return df1

In [14]:
df = process_isikumaarused(MAARUS_STATUS, DEPREL)

In [16]:
df = pd.read_csv(f"isikumaarused_{maarus_status2}_verb_patterns.csv", sep=",", encoding="utf-8")
df = df[~df["verb_word"].isna()]

In [18]:
# verbimustrite andmebaasi loomine/ühendumine
con = sqlite3.connect("verb_patterns_isikud.db")
cur = con.cursor()
#cur.execute('pragma encoding=UTF8') # loomise ajaks

In [19]:
df = df.fillna('')

In [20]:
# üheliikmelised mustrid
df

,pat_id,verbobl,word,case,gov,verb_word,compound_prt1,compound_prt2,compound_prt3,adp,other,verb,deprel
0,0,minema - abl (kellelt/millelt),minema,abl,kellelt/millelt,minema,,,,,,,obl
1,1,jääma - abl (kellelt/millelt),jääma,abl,kellelt/millelt,jääma,,,,,,,obl
2,2,lahkuma - abl (kellelt/millelt),lahkuma,abl,kellelt/millelt,lahkuma,,,,,,,obl
3,3,saabuma - abl (kellelt/millelt),saabuma,abl,kellelt/millelt,saabuma,,,,,,,obl
4,4,olema - abl (kellelt/millelt),olema,abl,kellelt/millelt,olema,,,,,,,obl
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6666,6666,musitseerima - in (kelles/milles),musitseerima,in,kelles/milles,musitseerima,,,,,,,obl
6667,6667,kõigutama - in (kelles/milles),kõigutama,in,kelles/milles,kõigutama,,,,,,,obl
6668,6668,kätlema - in (kelles/milles),kätlema,in,kelles/milles,kätlema,,,,,,,obl
6669,6669,kõmmutama - in (kelles/milles),kõmmutama,in,kelles/milles,kõmmutama,,,,,,,obl


In [22]:
cur.execute("""
DROP TABLE IF EXISTS {name}
""".format(name=pat_table_name))

In [21]:
pat_table_name

'patterns_isikud_len1_mitte_kunagi'

In [23]:
# andmebaasi tabelite loomine
cur.execute("CREATE TABLE {tablename}(ID INTEGER PRIMARY KEY, word TEXT, government TEXT, verb_word TEXT, compound_prt1 TEXT, compound_prt2 TEXT, compound_prt3 TEXT, w_case TEXT, adp TEXT, verb TEXT, other TEXT, deprel TEXT)".format(tablename=pat_table_name))

### vaja muuta käsitsi tabeli nime

In [24]:
%%time
for idx, row in df.iterrows():
    cur.execute("""INSERT INTO patterns_isikud_len1_mitte_kunagi
                          (word, government, verb_word, compound_prt1, compound_prt2, compound_prt3, w_case, adp, verb, other, deprel) 
                          VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);""", (row['word'], row['gov'], row['verb_word'], row['compound_prt1'], row['compound_prt2'], row['compound_prt3'], row['case'], row['adp'], row['verb'], row['other'], row['deprel']))
    con.commit()

CPU times: user 1.13 s, sys: 1.03 s, total: 2.16 s
Wall time: 39 s


In [25]:
con.close()

### vaja viia kujule

pattern_id, verb, +verb, match-type, kääne, nr, lisainfo(vms)

pat_id - (mustri ID tabelis patterns_len1)

pattern - (algne muster sõnena; NB! HETKEL ON SELLE VEERU READ MINGIL PÕHJUSEL NIHKES, SEETÕTTU PALUN HETKEL SEDA 

IGNOREERIDA, PARANDAN HILJEM)

verb_word - (mustri (pea)verb)

verb_compound - (pikema verbiühendi ülejäänud osad)

phrase_nr - (fraasi number; kuna hetkel on vaatluse all ainult tabelist patterns_len1 pärit mustrid, on kõigil fraasidel number 1)

phrase_case - (fraasi põhiliikme (pärast verbi) kääne; hiljem vaatame ilmselt vaid fraase, kus selles käändes on obliikva, kuid praeguseks pole seda tingimust veel sisse pandud)

adp - (kaassõna)

inf_verb - (infiniitverb)
